# 05 · Remaining life, pit window and counterfactuals

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

from boxbox.config import load_config
from boxbox.paths import artifact_paths, dataset_paths
from boxbox.preprocessing.build import read_lap_dataset

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4)
COLOURS = {"SOFT": "#c93a55", "MEDIUM": "#b07c00", "HARD": "#3b6fb0", "UNKNOWN": "#77766f"}
subset, full = load_config("subset"), load_config("full")

Strategy maths runs on exported artifacts through the same functions the API uses.

In [ ]:
import joblib
from boxbox.strategy.pit_loss import stop_losses
from boxbox.strategy.pit_window import StrategyInputs, choose_replacement, counterfactual, evaluate_pit_window
from boxbox.strategy.remaining_life import remaining_competitive_laps
from boxbox.strategy.uncertainty import SigmaCurve
paths = artifact_paths('full')
manifest = json.loads(paths.manifest.read_text())
model_name = manifest['selected_model']
model = joblib.load(paths.models / f'{model_name}.joblib')
s = manifest['sigma'][model_name]
sigma = SigmaCurve(tuple(s['per_lap']), tuple(s['cumulative']), s['power_a'], s['power_b'])
laps = pd.read_parquet(paths.data / 'laps.parquet')
print(model_name, manifest['pit_loss'])

In [ ]:
data = read_lap_dataset(full)
losses = stop_losses(data.laps[data.laps.race_id.isin(manifest['pit_loss']['races'])])
fig, ax = plt.subplots()
losses.loss_s.clip(10, 40).hist(bins=30, ax=ax, color='#2a78d6')
ax.axvline(manifest['pit_loss']['median_s'], color='k', ls='--')
ax.set(xlabel='green-flag pit loss (s)', title='Barcelona pit loss');

In [ ]:
driver = laps[(laps.race_id == '2024_10') & (laps.driver_id == 'alonso')].sort_values('lap')
state = driver[driver.lap == 12].iloc[0]
compound, feasible = choose_replacement(driver, 12)
inputs = StrategyInputs(compound, feasible, manifest['pit_loss']['median_s'], 0.0, 1.0, full.fuel_s_per_lap)
result = evaluate_pit_window(model, state, sigma, inputs)
life = remaining_competitive_laps(model, state, 1.0, 3, full.fuel_s_per_lap)
print('actual stops:', driver[driver.pit_in].lap.tolist(), '| window:', result.window, '| optimal:', result.optimal_lap, '| remaining laps:', life.value)

In [ ]:
cost = pd.DataFrame([c.__dict__ for c in result.candidates])
fig, ax = plt.subplots()
ax.plot(cost.pit_lap, cost.cost_pit_s - cost.cost_pit_s.min(), color='#2a78d6', label='cost vs best')
ax.plot(cost.pit_lap, cost.margin_s, color='grey', ls='--', label='margin')
if result.window: ax.axvspan(result.window.start_lap, result.window.end_lap, alpha=0.15)
for lap in driver[driver.pit_in].lap: ax.axvline(lap, color='#c93a55', ls=':')
ax.set(ylim=(0, 40), xlabel='pit lap', ylabel='s', title='Pit cost curve, Alonso 2024 lap 12'); ax.legend();

In [ ]:
cf = counterfactual(model, state, result.optimal_lap or 20, inputs)
fig, ax = plt.subplots()
ax.plot(cf.laps, cf.cumulative_delta_s); ax.axhline(0, color='grey')
ax.set(xlabel='lap', ylabel='pit − stay (cumulative s)', title='Counterfactual');

Warm-up sensitivity (D10): window at 0 / 0.5 / 1.0 s extra out-lap cost.

In [ ]:
pd.DataFrame([{'warmup_s': w, 'window': evaluate_pit_window(model, state, sigma, StrategyInputs(compound, feasible, inputs.pit_loss_s, w, 1.0, full.fuel_s_per_lap)).window} for w in (0.0, 0.5, 1.0)])

In [ ]:
pd.read_parquet(paths.eval / 'threshold_sensitivity.parquet').pivot(index='threshold_s', columns='model', values='mae_laps').round(2)

In [ ]:
timing = pd.read_parquet(paths.eval / 'pit_timing.parquet')
print(f"actual stop inside window (descriptive): {timing.actual_in_window.mean():.1%} of {len(timing)} stints")
timing.head(15)